In [12]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp

In [13]:
# Activation Functions
def relu(x):
    return np.maximum(0, x)

def deriv_relu(x):
    return np.where(x > 0, 1, 0)

def softmax(x):
    # Subtract max for stability, but do it per sample (axis=1)
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / np.sum(e_x, axis=1, keepdims=True)

def deriv_softmax(x):
    s = softmax(x)
    return s * (1 - s)

In [14]:
# Initialize parameters (Using He initialization as we are using ReLU as activation function)
def He_initialization(layer_sizes):
    weights = []
    biases = []
    for i in range(len(layer_sizes) - 1):
        weight = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2 / layer_sizes[i])
        bias = np.zeros((1, layer_sizes[i+1]))
        weights.append(weight)
        biases.append(bias)
    return weights, biases

In [15]:
#Regularization
def l2_regularization(weights, lambda_reg):
    l2_loss = 0
    for w in weights:
        l2_loss += np.sum(np.square(w))
    return (lambda_reg / 2) * l2_loss

In [16]:
# Loss Function
def cross_entropy(predictions, targets):
    n_samples = targets.shape[0]
    # Add a tiny epsilon (1e-15) to prevent log(0)
    predictions = np.clip(predictions, 1e-15, 1.0)
    logp = -np.log(predictions[range(n_samples), targets.argmax(axis=1)])
    loss_ce = np.sum(logp) / n_samples
    return loss_ce

def deriv_cross_entropy(predictions, targets):
    n_samples = targets.shape[0]
    grad = predictions - targets
    grad /= n_samples
    return grad

In [17]:
# Optimization functions
def sgd(weights, biases, weight_grads, bias_grads, learning_rate = 0.01):
    for i in range(len(weights)):
        weights[i] -= learning_rate * weight_grads[i]
        biases[i] -= learning_rate * bias_grads[i]
    return weights, biases

In [18]:
# Learning Rate decay
def decay_learning_rate(initial_lr, epoch, decay_rate=0.1, decay_step=10):
    return initial_lr * (decay_rate ** (epoch // decay_step))

In [19]:
# Neural Network Class
class SimpleNN:
    def __init__(self, layer):
        self.layer = layer
        self.weights = []
        self.biases = []
        self._init_params()

    def _init_params(self):
        self.weights, self.biases = He_initialization(self.layer)

    def forward_propagation(self, X):
        # store Zs and Activations for backpropagation
        self.Zs = []
        self.Activations = [X]

        A = X
        for i in range(len(self.weights)):
            Z = np.dot(A, self.weights[i]) + self.biases[i]
            self.Zs.append(Z)

            if i == len(self.weights) - 1: # output layer
                A = softmax(Z)
            else: # hidden layers
                A = relu(Z)

            self.Activations.append(A)
        return A 

    def back_propagation(self, X, Y, lambda_reg=0.01):
        m = Y.shape[0] 
        grads_w = [0] * len(self.weights) # gradients for weights
        grads_b = [0] * len(self.biases) # gradients for biases

        # Output layer gradient
        delta = deriv_cross_entropy(self.Activations[-1], Y)
        grads_w[-1] = np.dot(self.Activations[-2].T, delta) / m + (lambda_reg / m) * self.weights[-1] # L2 regularization term
        grads_b[-1] = np.sum(delta, axis=0, keepdims=True) / m + (lambda_reg / m) * self.biases[-1] # L2 regularization term

        # Hidden layers gradients
        for i in range(len(self.weights) - 2, -1, -1):
            delta = np.dot(delta, self.weights[i+1].T) * deriv_relu(self.Zs[i])
            grads_w[i] = np.dot(self.Activations[i].T, delta) / m + (lambda_reg / m) * self.weights[i] # L2 regularization term
            grads_b[i] = np.sum(delta, axis=0, keepdims=True) / m + (lambda_reg / m) * self.biases[i] # L2 regularization term

        return grads_w, grads_b

    def update_params(self, grads_w, grads_b, initial_learning_rate, epoch):
        learning_rate =  decay_learning_rate(initial_learning_rate, epoch)  # epoch can be passed if needed
        
        # Gradient Clipping to prevent overflow
        for i in range(len(grads_w)):
            grads_w[i] = np.clip(grads_w[i], -1, 1)
            grads_b[i] = np.clip(grads_b[i], -1, 1)
            
        self.weights, self.biases = sgd(self.weights, self.biases, grads_w, grads_b, learning_rate)

    def train(self, X, Y, epochs=20, initial_learning_rate=0.01, batch_size=32, lambda_reg=0.01):
        n_samples = X.shape[0]
        
        for epoch in range(epochs):
            # Shuffle the data at the start of each epoch for better generalization
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            Y_shuffled = Y[indices]
            
            epoch_loss = 0
            
            # Iterate through mini-batches -> Increases accuracy and convergence
            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i : i + batch_size]
                Y_batch = Y_shuffled[i : i + batch_size]
                
                # Forward propagation on the batch
                predictions = self.forward_propagation(X_batch)
                
                # Compute loss for the batch
                batch_loss = cross_entropy(predictions, Y_batch) + l2_regularization(self.weights, lambda_reg) # Include regularization in loss
                epoch_loss += batch_loss * (X_batch.shape[0] / n_samples)
                
                # Backward propagation on the batch
                grads_w, grads_b = self.back_propagation(X_batch, Y_batch, lambda_reg)
                
                # Update parameters (SGD now happens per batch)
                self.update_params(grads_w, grads_b, initial_learning_rate, epoch)

            # Calculate accuracy on full training set or last batch for tracking
            if epoch % 1 == 0: # Changed from 100 to 1 to see progress
                # For efficiency, we just use the last batch's predictions for accuracy here
                predicted_classes = np.argmax(predictions, axis=1)
                true_classes = np.argmax(Y_batch, axis=1)
                accuracy = np.mean(predicted_classes == true_classes)
                print(f'Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Batch Accuracy: {accuracy*100:.2f}%')



    def validate(self, X, Y):
        predictions = self.forward_propagation(X)
        loss = cross_entropy(predictions, Y)
        predicted_classes = np.argmax(predictions, axis=1)
        true_classes = np.argmax(Y, axis=1)
        accuracy = np.mean(predicted_classes == true_classes)
        print(f'Validation Loss: {loss:.4f}, Accuracy: {accuracy*100:.2f}%')

    def predict(self, X):
        predictions = self.forward_propagation(X)
        return np.argmax(predictions, axis=1)

In [20]:
from sklearn.datasets import fetch_openml
import numpy as np

mnist = fetch_openml('mnist_784', as_frame=False)
X, y = mnist['data'], mnist['target'].astype(int)

print(X.shape, y.shape)


(70000, 784) (70000,)


In [21]:
X = X / 255.0  # scale pixel values to [0,1]

# One-hot encoding of labels
def one_hot(labels, num_classes=10):
    return np.eye(num_classes)[labels]

y_onehot = one_hot(y)


In [22]:
np.random.seed(42)
indices = np.random.permutation(X.shape[0])

# Train: first 50k
X_train = X[indices[:50000]]
y_train = y_onehot[indices[:50000]]

# Validation: next 10k
X_val = X[indices[50000:60000]]
y_val = y_onehot[indices[50000:60000]]

# Test: last 10k
X_test = X[indices[60000:]]
y_test = y_onehot[indices[60000:]]

print(X_train.shape, y_train.shape)  
print(X_val.shape, y_val.shape)     
print(X_test.shape, y_test.shape)    



(50000, 784) (50000, 10)
(10000, 784) (10000, 10)
(10000, 784) (10000, 10)


In [23]:
nn = SimpleNN(layer=[784, 512, 256, 128, 10])
nn.train(X_train, y_train, epochs=20, initial_learning_rate=0.01, batch_size=64, lambda_reg=0.01)

Epoch 1, Loss: 11.3524, Batch Accuracy: 12.50%
Epoch 2, Loss: 11.1877, Batch Accuracy: 31.25%
Epoch 3, Loss: 11.0494, Batch Accuracy: 56.25%
Epoch 4, Loss: 10.9127, Batch Accuracy: 62.50%
Epoch 5, Loss: 10.7719, Batch Accuracy: 75.00%
Epoch 6, Loss: 10.6285, Batch Accuracy: 68.75%
Epoch 7, Loss: 10.4850, Batch Accuracy: 81.25%
Epoch 8, Loss: 10.3450, Batch Accuracy: 68.75%
Epoch 9, Loss: 10.2119, Batch Accuracy: 81.25%
Epoch 10, Loss: 10.0882, Batch Accuracy: 62.50%
Epoch 11, Loss: 10.0239, Batch Accuracy: 56.25%
Epoch 12, Loss: 10.0127, Batch Accuracy: 81.25%
Epoch 13, Loss: 10.0017, Batch Accuracy: 87.50%
Epoch 14, Loss: 9.9907, Batch Accuracy: 93.75%
Epoch 15, Loss: 9.9799, Batch Accuracy: 68.75%
Epoch 16, Loss: 9.9692, Batch Accuracy: 75.00%
Epoch 17, Loss: 9.9587, Batch Accuracy: 75.00%
Epoch 18, Loss: 9.9482, Batch Accuracy: 93.75%
Epoch 19, Loss: 9.9379, Batch Accuracy: 81.25%
Epoch 20, Loss: 9.9277, Batch Accuracy: 75.00%


In [24]:
nn.validate(X_val, y_val)

Validation Loss: 1.0689, Accuracy: 79.19%


In [25]:
# Optimized vectorized method
y_pred = nn.predict(X_test)                # Get all predictions at once
y_true = np.argmax(y_test, axis=1)         # Convert one-hot labels to class indices

accuracy = np.mean(y_pred == y_true)       # Calculate the ratio of matches
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 80.24%


Accuracy improved from 12.08% to 85.34% 